In [ ]:
import os
import subprocess
import sys
import traceback
import json
import gc
import re

print("=" * 60)
print("NEMOTRON LORA TRAINING WITH SFT - v23")
print("High-Fidelity CoT Distillation (250 Samples) + Rank 64 LoRA")
print("=" * 60)

try:
    # 1. Blackwell Environment Setup
    print("\n[1/8] Setting up Blackwell environment...")
    UTILITY_PATH = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script"
    if os.path.exists(UTILITY_PATH):
        subprocess.run(f"tar -cf - -C {UTILITY_PATH} . | tar -xf - -C /tmp", shell=True, check=True)
        for binary in ["ptxas", "ptxas-blackwell"]:
            bin_path = f"/tmp/triton/backends/nvidia/bin/{binary}"
            if os.path.exists(bin_path):
                subprocess.run(f"chmod +x {bin_path}", shell=True, check=True)
        os.environ["TRITON_PTXAS_PATH"] = "/tmp/triton/backends/nvidia/bin/ptxas-blackwell"
        sys.path.insert(0, "/tmp")
        print("Blackwell environment initialized")
    else:
        print(f"WARNING: Utility script not found")

    # 2. Imports & Dependencies
    print("\n[2/8] Loading dependencies...")
    
    MANDATORY_PACKAGES = ["trl", "peft", "bitsandbytes", "accelerate", "nvidia-cutlass", "mamba_ssm", "causal_conv1d"]
    print(f"Verifying mandatory packages: {MANDATORY_PACKAGES}")
    
    for pkg in MANDATORY_PACKAGES:
        try:
            __import__(pkg.replace("-", "_"))
            print(f"  {pkg} already installed")
        except ImportError:
            print(f"  Installing {pkg}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", pkg])

    import torch
    import pandas as pd
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        TrainingArguments,
        BitsAndBytesConfig
    )
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from datasets import Dataset
    from trl import SFTTrainer
    import kagglehub

    # 3. Hardware check
    print(f"\n[3/8] Hardware configuration:")
    print(f"  PyTorch: {torch.__version__}")
    print(f"  CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            prop = torch.cuda.get_device_properties(i)
            print(f"  GPU {i}: {prop.name} ({prop.total_memory / 1024**3:.1f} GB)")

    # 4. Load competition data
    print("\n[4/8] Loading training data...")
    competition_id = "nvidia-nemotron-model-reasoning-challenge"

    train_file = None
    for base_path in [f"/kaggle/input/{competition_id}", "/kaggle/input"]:
        if os.path.exists(base_path):
            for root, dirs, files in os.walk(base_path):
                for f in files:
                    if "train" in f.lower() and f.endswith(".csv"):
                        train_file = os.path.join(root, f)
                        break
                if train_file: break
        if train_file: break

    if not train_file:
        print("ERROR: Training data not found")
        sys.exit(1)

    df = pd.read_csv(train_file)
    print(f"  Dataset: {train_file} ({len(df)} samples)")

    # 5. Teacher trace generation
    print("\n[5/8] Generating teacher traces...")
    teacher_model_name = "deepseek-ai/deepseek-r1-distill-qwen-32b"

    try:
        print(f"  Loading teacher: {teacher_model_name}")
        teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_model_name, trust_remote_code=True)
        teacher_model = AutoModelForCausalLM.from_pretrained(
            teacher_model_name,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True
        )
        
        def extract_boxed_answer(text):
            if not text: return None
            matches = re.findall(r"\\boxed\{([^}]+)\}", str(text))
            if matches:
                inner = matches[-1]
                nums = re.findall(r"-?\d+", inner)
                return int(nums[-1]) if nums else None
            nums = re.findall(r"-?\d+", str(text))
            return int(nums[-1]) if nums else None

        def generate_trace(problem):
            prompt = f"<extra_id_1>User\nSolve this step by step and put your final answer in \\boxed{{}}.\n\nProblem: {problem}\n<extra_id_1>Assistant\n<think>\n"
            inputs = teacher_tokenizer(prompt, return_tensors="pt").to(teacher_model.device)
            with torch.no_grad():
                outputs = teacher_model.generate(
                    **inputs,
                    max_new_tokens=1536,
                    temperature=0.6,
                    do_sample=True,
                    pad_token_id=teacher_tokenizer.eos_token_id
                )
            return teacher_tokenizer.decode(outputs[0], skip_special_tokens=True)[len(prompt):].strip()

        sample_size = min(250, len(df))
        print(f"  Processing {sample_size} samples...")

        filtered_data = []
        for idx in range(sample_size):
            row = df.iloc[idx]
            problem = row.get("problem", row.get("question", row.get("prompt")))
            gt_str = str(row.get("answer", row.get("expected_answer", row.get("response"))))
            gt = extract_boxed_answer(gt_str)
            
            trace = generate_trace(problem)
            teacher_ans = extract_boxed_answer(trace)
            
            if teacher_ans == gt and gt is not None:
                filtered_data.append({"problem": problem, "answer": gt, "teacher_trace": trace})
            
            if (idx + 1) % 25 == 0:
                print(f"    {idx + 1}/{sample_size} (Kept {len(filtered_data)})")

        del teacher_model
        gc.collect()
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"  Teacher distillation failed: {e}")
        filtered_data = []
        for _, row in df.head(100).iterrows():
            p = row.get("problem", row.get("question", row.get("prompt")))
            a_str = str(row.get("answer", row.get("expected_answer", row.get("response"))))
            a = extract_boxed_answer(a_str) or 0
            filtered_data.append({"problem": p, "answer": a, "teacher_trace": ""})

    # 6. Student model configuration
    print("\n[6/8] Loading student model...")
    model_id = "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
    model_path = kagglehub.model_download(model_id)
    os.makedirs("/tmp/offload", exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
        offload_folder="/tmp/offload"
    )

    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "in_proj", "out_proj", "x_proj", "dt_proj", "w1", "w2", "w3"]
    lora_config = LoraConfig(
        r=64,
        lora_alpha=128,
        target_modules=target_modules,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # 7. Dataset Prep
    print("\n[7/8] Preparing dataset...")
    def format_text(ex):
        if ex["teacher_trace"]:
            return {"text": f"Problem: {ex[problem]}\n\n<think>\n{ex[teacher_trace]}\n</think>\nAnswer: {ex[answer]}"}
        return {"text": f"Problem: {ex[problem]}\n\nAnswer: {ex[answer]}"}

    dataset = Dataset.from_list(filtered_data).map(format_text)
    dataset = dataset.map(lambda x: {"len": len(x["text"])})
    dataset = dataset.sort("len")
    
    split = dataset.train_test_split(test_size=0.1)

    # 8. Training Loop
    print("\n[8/8] Starting training...")
    training_args = TrainingArguments(
        output_dir="./nemotron_lora_adapter",
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=1e-4,
        warmup_ratio=0.1,
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=25,
        save_strategy="no",
        bf16=True,
        gradient_checkpointing=True,
        report_to="none",
        remove_unused_columns=False,
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=split["train"],
        eval_dataset=split["test"],
        args=training_args,
        max_seq_length=2048,
    )
    
    trainer.train()
    
    print("\nSaving adapter and packaging for submission...")
    trainer.save_model("./nemotron_lora_adapter")
    tokenizer.save_pretrained("./nemotron_lora_adapter")
    subprocess.run("cd nemotron_lora_adapter && zip -r ../submission.zip ./*", shell=True, check=True)
    print("\nDONE.")

except Exception as e:
    print(f"\nFATAL ERROR: {e}")
    traceback.print_exc()
    sys.exit(1)
